# Lab 9 – Local Binary Pattern and Image Denoising

Implement Local Binary Pattern on a grayscale image. Then denoise Gaussian-noisy and speckle-noisy images using Wiener, median, Gaussian, and bilateral filters.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import wiener

image_path = '/content/pexels-chevanon-1108099.jpg'
image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)

if image is None:
    raise FileNotFoundError('Image not found. Check the image path.')

plt.imshow(image, cmap='gray')
plt.title('Original Grayscale Image')
plt.axis('off')
plt.show()

## 1. Local Binary Pattern (LBP)

In [ ]:
height, width = image.shape
lbp_image = np.zeros((height - 2, width - 2), dtype=np.uint8)

for i in range(1, height - 1):
    for j in range(1, width - 1):
        center = image[i, j]
        value = 0

        if image[i-1, j-1] >= center:
            value += 128
        if image[i-1, j] >= center:
            value += 64
        if image[i-1, j+1] >= center:
            value += 32
        if image[i, j+1] >= center:
            value += 16
        if image[i+1, j+1] >= center:
            value += 8
        if image[i+1, j] >= center:
            value += 4
        if image[i+1, j-1] >= center:
            value += 2
        if image[i, j-1] >= center:
            value += 1

        lbp_image[i-1, j-1] = value

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.imshow(image, cmap='gray')
plt.title('Original Grayscale Image')
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(lbp_image, cmap='gray')
plt.title('Local Binary Pattern')
plt.axis('off')
plt.show()

## 2. Denoising Gaussian and Speckle Noise

A 5×5 window is used for all four filters.

In [ ]:
# Add Gaussian noise
gaussian_noise = np.random.normal(0, 25, image.shape)
gaussian_noisy = np.clip(image.astype(np.float32) + gaussian_noise, 0, 255).astype(np.uint8)

# Add Speckle noise
speckle_noise = np.random.normal(0, 0.2, image.shape)
speckle_noisy = np.clip(image.astype(np.float32) + image.astype(np.float32) * speckle_noise, 0, 255).astype(np.uint8)

# Wiener filter
wiener_gaussian = np.clip(wiener(gaussian_noisy, (5, 5)), 0, 255).astype(np.uint8)
wiener_speckle = np.clip(wiener(speckle_noisy, (5, 5)), 0, 255).astype(np.uint8)

# Median filter
median_gaussian = cv2.medianBlur(gaussian_noisy, 5)
median_speckle = cv2.medianBlur(speckle_noisy, 5)

# Gaussian filter
gau_filter_gaussian = cv2.GaussianBlur(gaussian_noisy, (5, 5), 1.5)
gau_filter_speckle = cv2.GaussianBlur(speckle_noisy, (5, 5), 1.5)

# Bilateral filter
bilateral_gaussian = cv2.bilateralFilter(gaussian_noisy, 5, 75, 75)
bilateral_speckle = cv2.bilateralFilter(speckle_noisy, 5, 75, 75)

## Display the Denoising Results

In [ ]:
titles = ['Noisy Image', 'Wiener Filter', 'Median Filter', 'Gaussian Filter', 'Bilateral Filter']
gaussian_results = [gaussian_noisy, wiener_gaussian, median_gaussian, gau_filter_gaussian, bilateral_gaussian]
speckle_results = [speckle_noisy, wiener_speckle, median_speckle, gau_filter_speckle, bilateral_speckle]

plt.figure(figsize=(15, 6))
for i in range(5):
    plt.subplot(2, 5, i + 1)
    plt.imshow(gaussian_results[i], cmap='gray')
    plt.title('Gaussian ' + titles[i])
    plt.axis('off')

    plt.subplot(2, 5, i + 6)
    plt.imshow(speckle_results[i], cmap='gray')
    plt.title('Speckle ' + titles[i])
    plt.axis('off')

plt.tight_layout()
plt.show()